# Scientific PDF → AI Knowledge
**V3.1 Autorecover | Google Colab | PDF + optional JSON/JSONL**

This notebook is a user guide and execution interface. All implementation code lives in `src/` on Drive. The original extraction engine remains unchanged. Select **Runtime → Change runtime type → T4 GPU** for PDF extraction; JSON conversion uses CPU only.

## 1. Mount Drive and select the workflow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/PDF_AI_Project')  # change for your own workspace
SRC = ROOT / 'src'  # copy repository src/ here, or point to a local clone
PDF_INPUT = ROOT / 'pdfs'
PDF_OUTPUT = ROOT / 'knowledge'
for folder in (PDF_INPUT, PDF_OUTPUT): folder.mkdir(parents=True, exist_ok=True)
ONLY = []  # all PDFs, or ['Your_Book.pdf']
RUN_PDF = False
RUN_JSON = False
print('Available PDFs:', [p.name for p in sorted(PDF_INPUT.glob('*.pdf'))])

## 2. Install dependencies (PDF only)
Set `INSTALL_PDF_DEPS=True` on a fresh runtime. When resuming, the installer selects the saved Docling version. JSON-only runs do not need these packages.

In [ ]:
import os, sys, json, subprocess, shutil, importlib.metadata
INSTALL_PDF_DEPS = False  # set True on a fresh PDF runtime
profiles = []
for p in sorted(PDF_INPUT.glob('*.pdf')):
    if ONLY and p.name.casefold() not in {x.casefold() for x in ONLY} and p.stem.casefold() not in {x.casefold() for x in ONLY}: continue
    f = PDF_OUTPUT / p.stem / 'pipeline_profile.json'
    if f.exists(): profiles.append(json.loads(f.read_text()))
versions = {p['docling_version'] for p in profiles}
if INSTALL_PDF_DEPS:
    if len(versions)>1: raise RuntimeError('Select books with one saved Docling version at a time')
    spec = 'docling==' + next(iter(versions)) if versions else 'docling'
    subprocess.run([sys.executable,'-m','pip','install',spec,'sentence-transformers','pypdf','requests==2.32.4'],check=True)
import psutil
print('CPU threads:', os.cpu_count(), 'RAM GiB:', round(psutil.virtual_memory().total/2**30,1))
print('Free local GiB:', round(shutil.disk_usage('/content').free/2**30,1))
try:
    import torch
    print('CUDA:', torch.cuda.is_available())
    if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
    print('Docling:', importlib.metadata.version('docling'))
except (ImportError, importlib.metadata.PackageNotFoundError) as e: print(e)
print('Saved profile Docling versions:', versions)

## 3. Load Python libraries
Copy the source modules to local Colab storage for execution. The notebook contains no embedded implementation source. Run this section after every runtime restart.

In [ ]:
import sys, shutil
LOCAL = Path('/content/pdf_ai_src')
LOCAL.mkdir(exist_ok=True)
for name in ('convert_books_v3_1_autorecover.py', 'json_to_ai.py', 'audit_extraction.py'):
    shutil.copy2(SRC / name, LOCAL / name)
sys.path.insert(0, str(LOCAL))
ENGINE = LOCAL / 'convert_books_v3_1_autorecover.py'
import json_to_ai

## 4. PDF configuration and execution
The default is the fastest observed T4 configuration supplied by the user. `MAX_BATCHES=None` means full extraction; set 1 only for a test in a separate output directory. Resume with the exact same profile (including threads, device, formula flag, scale and Docling version). A device change requires a new output directory. The engine reassembles the book and global RAG index after a run, so protect any manually edited root indexes first. A selected-book run may replace the global index with only the selected books; use per-book indexes as the authoritative retrieval sources. The wrapper backs up existing root publications to a separate directory before running, without touching batch checkpoints.

In [ ]:
DEVICE = 'cuda'
BATCH_PAGES = 10
THREADS = 8
BATCHES_PER_WORKER = 3
MAX_TOKENS = 220
IMAGE_SCALE = 2.0
FORMULA = True
BACKEND = 'pdfium'
MAX_BATCHES = None
command = [sys.executable,'-u',str(ENGINE),'--input',str(PDF_INPUT),'--output',str(PDF_OUTPUT),
    '--batch-pages',str(BATCH_PAGES),'--threads',str(THREADS),'--device',DEVICE,
    '--batches-per-worker',str(BATCHES_PER_WORKER),'--max-tokens',str(MAX_TOKENS),
    '--image-scale',str(IMAGE_SCALE),'--backend-strategy',BACKEND]
if ONLY: command += ['--only',*ONLY]
if FORMULA: command += ['--formula']
if MAX_BATCHES is not None: command += ['--max-batches',str(MAX_BATCHES)]
import shlex
print(shlex.join(command))
if RUN_PDF:
    if not list(PDF_INPUT.glob('*.pdf')): raise RuntimeError('Upload PDFs to the pdfs folder first')
    import torch
    if DEVICE == 'cuda' and not torch.cuda.is_available(): raise RuntimeError('Select GPU runtime')
    from datetime import datetime, timezone
    backup = ROOT / 'publication_backups' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    names = {'book.md','quality_report.md','rag_chunks.jsonl','tables_index.jsonl','figures_to_review.jsonl','_COMPLETE.json'}
    for f in [PDF_OUTPUT/'all_rag_chunks.jsonl', *(f for d in PDF_OUTPUT.iterdir() if d.is_dir() and not d.name.startswith('_') for f in d.iterdir() if f.is_file() and f.name in names)]:
        if f.is_file():
            target=backup/f.relative_to(PDF_OUTPUT); target.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(f,target)
    subprocess.run(command,check=True)
else: print('Preview only. Set RUN_PDF=True in section 1, then rerun this cell.')

## 5. Audit existing extraction without running Docling
Checks exact source hash, physical page coverage, checkpoint file hashes, and book-level index consistency against batch-level records. Root figure/table indexes may contain valuable manually reviewed descriptions; differences are reported rather than overwritten. No files are changed. This audit verifies integrity and aggregation, not scientific accuracy.

In [ ]:
from audit_extraction import audit_book
AUDIT_BOOK = None  # e.g. PDF_OUTPUT / 'Your_Book'
if AUDIT_BOOK is not None:
    audit_book(AUDIT_BOOK, PDF_INPUT)

## 6. Optional JSON / JSONL → Markdown
No PDF processing. The converter validates JSON syntax independently and preserves duplicate keys, field order, literal number tokens, escape sequences and record order. JSONL records retain original line numbers. A nested output is supported; generated directories are excluded from scanning. Output uses deterministic content-addressed runs, verified readback, manifests and a `LATEST.json` pointer updated only on complete success. TXT is optional and off. Operational checkpoints and detailed structure are excluded by default; opt in when necessary. The default includes both root and batch indexes because they may differ. After auditing their equality, enable canonical mode to suppress batch index copies. Use a mounted Drive folder; for shared folders add a Drive shortcut first if needed. Native Google Docs publication is not included in this version.

In [ ]:
JSON_INPUT = None  # e.g. PDF_OUTPUT / 'Your_Book' or any JSON folder
DETAILED_STRUCTURE = False
INCLUDE_ADMIN = False
CANONICAL_INDEXES = False  # enable only after comparing root and batch indexes
WRITE_TXT = False
if JSON_INPUT is not None:
    JSON_OUTPUT = JSON_INPUT / 'AI_KNOWLEDGE'
    json_to_ai.convert(JSON_INPUT, JSON_OUTPUT, detailed=DETAILED_STRUCTURE,
        include_admin=INCLUDE_ADMIN, canonical=CANONICAL_INDEXES,
        write_txt=WRITE_TXT, dry_run=not RUN_JSON)
elif RUN_JSON:
    raise ValueError('Set JSON_INPUT first')

## 7. AI knowledge and recovery guide
Use `book.md`, `quality_report.md`, relevant tables and actual figure images. For custom retrieval use per-book `rag_chunks.jsonl`. JSON Markdown is optional; avoid indexing duplicate representations. Raw batches and completion markers are required for checkpoint validation. Keep the original PDF available elsewhere if you need to verify or resume an old extraction; `pdfs/` starts empty. `SRBIAU/` is not used by this notebook.

- Resume with the same source bytes, Docling version and extraction settings. Never bypass checkpoint hashes.
- CUDA unavailable: select GPU or use a separate CPU output location.
- PDFium is the default; `auto` retries a SIGTRAP batch with PDFium. Review worker failures before retrying.
- Scientific accuracy: OCR is disabled in the existing engine; inspect complex formulas, tables, figures and sparse pages against the original. Checksums establish integrity, not accuracy.
- JSON invalid records: inspect manifest and correct the source deliberately; do not mark partial output complete.
- Keep PDFs, output, backups and private data out of Git.

The modular notebook is syntax-checked locally; a full GPU run in Colab is not performed here.